In [ ]:
import atexit
import hashlib
import json
import os
import threading
import time
from pathlib import Path

RUN_ID = "zebrahub-multiscale-contextual-pretrain-v1"
PRETRAINING_RUN_ID = "zebrahub-multiscale-contextual-pretrain-v1"
STARTED = time.monotonic()
FINISHED = False
TERMINAL = Path("/kaggle/working/launcher_terminal.json")
OUTPUT_DIR = Path("/kaggle/working/zebrahub_multiscale_contextual_pretrain_v1")
DECLARED_BUDGET_SECONDS = 24_000


def write_terminal(status, error=None):
    pretraining_terminal = OUTPUT_DIR / "pretraining_terminal.json"
    payload = {
        "schema_version": 1,
        "run_id": RUN_ID,
        "status": status,
        "elapsed_seconds": round(time.monotonic() - STARTED, 3),
        "declared_budget_seconds": DECLARED_BUDGET_SECONDS,
        "trainer_max_wall_seconds": 21_600,
        "trainer_hard_stop_seconds": 22_800,
        "pretraining_terminal_exists": pretraining_terminal.is_file(),
        "competition_data_read": False,
        "public_leaderboard_used_for_selection": False,
        "submission_created": False,
    }
    if error is not None:
        payload["error"] = str(error)
    if pretraining_terminal.is_file():
        payload["pretraining_terminal_sha256"] = hashlib.sha256(
            pretraining_terminal.read_bytes()
        ).hexdigest()
    temporary = TERMINAL.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    temporary.replace(TERMINAL)


def abort_terminal():
    if not FINISHED:
        write_terminal("aborted")


def budget_expired():
    write_terminal("budget_expired")
    os._exit(124)


atexit.register(abort_terminal)
TIMER = threading.Timer(DECLARED_BUDGET_SECONDS, budget_expired)
TIMER.daemon = True
TIMER.start()
print("ZebraHub contextual-pretraining watchdog armed for 24,000 seconds.")


# Project-authored multiscale contextual pretraining v1

The 46.4M-parameter residual expansion starts only from both strictly loaded v3 checkpoints bound to passing one-shot ZSNS001 acceptance. Two isolated GPUs train on ZSNS004 and select only on fixed ZSNS005 shards. No competition data, public predictions, leaderboard selection, or submission path is available.


In [ ]:
import shutil
import subprocess
import sys
import zipfile

import torch

EXPECTED_DATASET_MANIFEST_SHA256 = "b35738f215413f1ece403ba5c0601adea82e2540c65f37e6465de0d0755cb7bf"
EXPECTED_RUNTIME_MANIFEST_SHA256 = "ac1c32a70f3dcc699806d18bde487d5d9154ba6774f06c7a812773c0e0efb8b3"


def first_existing(candidates):
    return next((path for path in candidates if path.exists()), None)


def materialize_runtime_input(runtime_input, runtime):
    if runtime.exists():
        shutil.rmtree(runtime)
    runtime.mkdir(parents=True)
    for source in runtime_input.iterdir():
        if source.is_dir():
            shutil.copytree(source, runtime / source.name)
        elif source.is_file() and source.suffix != ".zip" and source.name != "dataset-metadata.json":
            shutil.copy2(source, runtime / source.name)
    for archive in runtime_input.glob("*.zip"):
        destination = runtime / archive.stem
        if destination.exists():
            raise RuntimeError(f"Ambiguous runtime directory and archive: {archive.name}")
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archive) as handle:
            handle.extractall(destination)


def materialize_shard_input(shard_input, shard_root):
    if shard_root.exists():
        shutil.rmtree(shard_root)
    shard_root.mkdir(parents=True)
    for source in shard_input.iterdir():
        if source.is_dir():
            shutil.copytree(source, shard_root / source.name)
        elif source.is_file() and source.suffix != ".zip" and source.name != "dataset-metadata.json":
            shutil.copy2(source, shard_root / source.name)
    for archive in sorted(shard_input.glob("*.zip")):
        with zipfile.ZipFile(archive) as handle:
            members = [name for name in handle.namelist() if name and not name.endswith("/")]
            roots = {Path(name).parts[0] for name in members}
            if archive.stem in {"train", "validation"} and roots != {archive.stem}:
                destination = shard_root / archive.stem
                if destination.exists():
                    raise RuntimeError(f"Ambiguous shard directory and archive: {archive.name}")
                destination.mkdir(parents=True)
            else:
                destination = shard_root
                if any((shard_root / root).exists() for root in roots):
                    raise RuntimeError(f"Ambiguous shard content and archive: {archive.name}")
            handle.extractall(destination)


if torch.cuda.device_count() != 2:
    raise RuntimeError(
        f"ZebraHub contextual pretraining requires exactly two GPUs, saw {torch.cuda.device_count()}"
    )
gpu_names = [torch.cuda.get_device_name(index) for index in range(2)]
print({"gpu_count": 2, "gpu_names": gpu_names})

input_root = Path("/kaggle/input")
runtime_input = first_existing([
    Path("/kaggle/input/datasets/indarkarhana/biohub-temporal-multiscale-contextual-runtime-v4"),
    Path("/kaggle/input/biohub-temporal-multiscale-contextual-runtime-v4"),
])
shard_input = first_existing([
    Path("/kaggle/input/datasets/indarkarhana/biohub-zebrahub-contextual-shards-v1"),
    Path("/kaggle/input/biohub-zebrahub-contextual-shards-v1"),
])
if runtime_input is None:
    runtime_input = next(
        (
            path.parent
            for path in input_root.rglob("SOURCE_MANIFEST.json")
            if json.loads(path.read_text(encoding="utf-8")).get("run_id")
            == "temporal-contextual-pair-fusion-v3"
        ),
        None,
    )
if shard_input is None:
    shard_input = next(
        (
            path.parent
            for path in input_root.rglob("DATASET_MANIFEST.json")
            if json.loads(path.read_text(encoding="utf-8")).get("run_id")
            == "zebrahub-contextual-shards-v1"
        ),
        None,
    )
if runtime_input is None or shard_input is None:
    raise FileNotFoundError({"runtime_input": runtime_input, "shard_input": shard_input})

runtime = Path("/kaggle/working/multiscale_contextual_runtime_v4")
shard_root = Path("/kaggle/working/zebrahub_contextual_shards_v1")
materialize_runtime_input(runtime_input, runtime)
materialize_shard_input(shard_input, shard_root)

runtime_manifest_sha256 = hashlib.sha256(
    (runtime / "SOURCE_MANIFEST.json").read_bytes()
).hexdigest()
if runtime_manifest_sha256 != EXPECTED_RUNTIME_MANIFEST_SHA256:
    raise RuntimeError(f"Runtime manifest changed: {runtime_manifest_sha256}")
subprocess.run(
    [
        sys.executable,
        str(runtime / "verify_runtime.py"),
        "--root",
        str(runtime),
        "--require-gpus",
    ],
    check=True,
)
verification = subprocess.run(
    [
        sys.executable,
        str(runtime / "verify_zebrahub_contextual_dataset.py"),
        "--root",
        str(shard_root),
    ],
    check=True,
    capture_output=True,
    text=True,
)
dataset_evidence = json.loads(verification.stdout)
if not (
    dataset_evidence.get("status") == "verified"
    and dataset_evidence.get("run_id") == "zebrahub-contextual-shards-v1"
    and dataset_evidence.get("manifest_sha256") == EXPECTED_DATASET_MANIFEST_SHA256
    and dataset_evidence.get("training", {}).get("shards") == 64
    and dataset_evidence.get("validation", {}).get("shards") == 16
    and dataset_evidence.get("competition_test_data_read") is False
    and dataset_evidence.get("public_competition_predictions_read") is False
    and dataset_evidence.get("leaderboard_used") is False
    and dataset_evidence.get("submission_created") is False
):
    raise RuntimeError(f"Invalid derived dataset evidence: {dataset_evidence}")

print(json.dumps({
    "runtime": str(runtime),
    "runtime_manifest_sha256": runtime_manifest_sha256,
    "shard_root": str(shard_root),
    "dataset_evidence": dataset_evidence,
}, indent=2, sort_keys=True))


# Bind the warm start to the independently accepted v3 output.
import math

EXPECTED_ACCEPTANCE_MANIFEST_SHA256 = "cbbf670dde160e5a927ed84bb9e2a7313abe4506f4798f6afa00680fc8e7c6d0"
EXPECTED_ACCEPTANCE_INVENTORY_SHA256 = "e32bc686e14222e43acb8d6247351e286eae8ed6fdb1f4ab5087e55fb0c79667"
EXPECTED_ACCEPTANCE_RECORD_INVENTORY_SHA256 = "05ad8b3195aa2786ffb8a2ffcb247d118d1e5cf96026264e0534c468979e6e0e"
ACCEPTANCE_FOLDS = ("target_44b6", "target_6bba")
ACCEPTANCE_SEEDS = {"target_44b6": 51004, "target_6bba": 61007}

def unique_json_parent(filename, predicate):
    matches = []
    for path in Path("/kaggle/input").rglob(filename):
        payload = json.loads(path.read_text(encoding="utf-8"))
        if predicate(payload, path.parent):
            matches.append(path.parent)
    if len(matches) != 1:
        raise RuntimeError({"filename": filename, "matches": [str(path) for path in matches]})
    return matches[0]


def sha256_file(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def checked_acceptance_metrics(payload, fold, stage):
    if not isinstance(payload, dict):
        raise RuntimeError(f"{fold} {stage} acceptance metrics are missing")
    result = {}
    for name in ("composite", "top1", "mrr", "division_top2"):
        value = payload.get(name)
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise RuntimeError(f"{fold} {stage} acceptance metric is malformed: {name}")
        value = float(value)
        if not math.isfinite(value) or not 0.0 <= value <= 1.0:
            raise RuntimeError(f"{fold} {stage} acceptance metric is outside [0, 1]: {name}")
        result[name] = value
    for name in ("rows", "division_rows", "transitions"):
        value = payload.get(name)
        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:
            raise RuntimeError(f"{fold} {stage} acceptance inventory is malformed: {name}")
        result[name] = value
    return result


def recompute_acceptance_gate(initial, final):
    gains = {
        name: float(final[name]) - float(initial[name])
        for name in ("composite", "top1", "mrr", "division_top2")
    }
    inventory_unchanged = all(
        int(final[name]) == int(initial[name])
        for name in ("rows", "division_rows", "transitions")
    )
    return {
        "passed": bool(
            inventory_unchanged
            and gains["composite"] >= 0.01
            and gains["top1"] > 0.0
            and gains["mrr"] > 0.0
            and gains["division_top2"] >= 0.0
        ),
        "minimum_composite_gain": 0.01,
        "inventory_unchanged": inventory_unchanged,
        "gains": gains,
    }



pretraining_root = unique_json_parent(
    "pretraining_terminal.json",
    lambda payload, parent: (
        payload.get("run_id") == "zebrahub-contextual-pretrain-v1"
        and payload.get("status") == "completed"
        and payload.get("gpu_count") == 2
        and payload.get("both_folds_improved") is True
        and (parent / "target_44b6" / "pretrained_model.pt").is_file()
        and (parent / "target_6bba" / "pretrained_model.pt").is_file()
    ),
)
acceptance_root = unique_json_parent(
    "acceptance_terminal.json",
    lambda payload, _parent: (
        payload.get("run_id") == "zebrahub-contextual-acceptance-evaluation-v1"
        and payload.get("status") == "completed"
        and payload.get("gpu_count") == 2
        and payload.get("both_folds_improved") is True
        and payload.get("acceptance_manifest_sha256")
        == EXPECTED_ACCEPTANCE_MANIFEST_SHA256
        and payload.get("acceptance_inventory_sha256")
        == EXPECTED_ACCEPTANCE_INVENTORY_SHA256
        and payload.get("acceptance_record_inventory_sha256")
        == EXPECTED_ACCEPTANCE_RECORD_INVENTORY_SHA256
        and payload.get("selection_or_checkpoint_redirect_permitted") is False
        and payload.get("competition_data_read") is False
        and payload.get("public_predictions_copied") is False
        and payload.get("public_leaderboard_used_for_selection") is False
        and payload.get("submission_created") is False
    ),
)

pretraining_terminal_path = pretraining_root / "pretraining_terminal.json"
pretraining_terminal_sha256 = hashlib.sha256(
    pretraining_terminal_path.read_bytes()
).hexdigest()
acceptance = json.loads(
    (acceptance_root / "acceptance_terminal.json").read_text(encoding="utf-8")
)
if acceptance.get("pretraining_terminal_sha256") != pretraining_terminal_sha256:
    raise RuntimeError("Acceptance and transfer pretraining roots diverge")
acceptance_folds = acceptance.get("folds")
if not isinstance(acceptance_folds, dict) or set(acceptance_folds) != set(ACCEPTANCE_FOLDS):
    raise RuntimeError("Acceptance aggregate fold inventory changed")
launcher_candidates = sorted(acceptance_root.parent.rglob("launcher_terminal.json"))
if len(launcher_candidates) != 1:
    raise RuntimeError(f"Expected one acceptance launcher terminal: {launcher_candidates}")
launcher = json.loads(launcher_candidates[0].read_text(encoding="utf-8"))
acceptance_terminal_sha256 = sha256_file(acceptance_root / "acceptance_terminal.json")
if not (
    launcher.get("schema_version") == 1
    and launcher.get("run_id") == "zebrahub-contextual-acceptance-evaluation-v1"
    and launcher.get("status") == "completed"
    and launcher.get("declared_budget_seconds") == 3600
    and launcher.get("evaluator_hard_stop_seconds") == 3300
    and launcher.get("acceptance_terminal_exists") is True
    and launcher.get("acceptance_terminal_sha256") == acceptance_terminal_sha256
    and launcher.get("gpu_count_required") == 2
    and launcher.get("competition_data_read") is False
    and launcher.get("public_predictions_copied") is False
    and launcher.get("public_leaderboard_used_for_selection") is False
    and launcher.get("submission_created") is False
):
    raise RuntimeError("Acceptance launcher terminal is invalid")
for fold in ACCEPTANCE_FOLDS:
    model_path = pretraining_root / fold / "pretrained_model.pt"
    model_sha256 = sha256_file(model_path)
    child_path = acceptance_root / fold / "acceptance_terminal.json"
    worker_path = pretraining_root / fold / "worker_terminal.json"
    if not child_path.is_file() or not worker_path.is_file():
        raise FileNotFoundError(f"Acceptance or pretraining fold evidence is missing: {fold}")
    child = json.loads(child_path.read_text(encoding="utf-8"))
    aggregate_row = dict(acceptance_folds[fold])
    if aggregate_row.pop("terminal_sha256", None) != sha256_file(child_path):
        raise RuntimeError(f"Acceptance aggregate does not bind child terminal: {fold}")
    if aggregate_row != child:
        raise RuntimeError(f"Acceptance aggregate and child terminal diverge: {fold}")
    initial = checked_acceptance_metrics(child.get("initial"), fold, "initial")
    final = checked_acceptance_metrics(child.get("final"), fold, "final")
    gate = recompute_acceptance_gate(initial, final)
    if not (
        child.get("schema_version") == 1
        and child.get("status") == "completed"
        and child.get("run_id") == "zebrahub-contextual-acceptance-evaluation-v1"
        and child.get("fold") == fold
        and child.get("seed") == ACCEPTANCE_SEEDS[fold]
        and child.get("appearance_family") == "temporal_contextual_pair_fusion_v3"
        and child.get("parameter_count") == 20_747_761
        and child.get("acceptance_source") == "ZSNS001"
        and child.get("acceptance_role") == "external_acceptance"
        and child.get("acceptance_shards") == 16
        and child.get("acceptance_manifest_sha256")
        == EXPECTED_ACCEPTANCE_MANIFEST_SHA256
        and child.get("acceptance_inventory_sha256")
        == EXPECTED_ACCEPTANCE_INVENTORY_SHA256
        and child.get("acceptance_record_inventory_sha256")
        == EXPECTED_ACCEPTANCE_RECORD_INVENTORY_SHA256
        and child.get("pretraining_terminal_sha256") == pretraining_terminal_sha256
        and child.get("pretrained_model_sha256") == model_sha256
        and child.get("pretraining_worker_terminal_sha256") == sha256_file(worker_path)
        and child.get("gate") == gate
        and child.get("gate_passed") is True
        and gate["passed"] is True
        and child.get("selection_or_checkpoint_redirect_permitted") is False
        and child.get("competition_data_read") is False
        and child.get("public_predictions_copied") is False
        and child.get("public_leaderboard_used_for_selection") is False
        and child.get("submission_created") is False
    ):
        raise RuntimeError(f"Acceptance fold evidence is invalid: {fold}")
    if acceptance_folds[fold].get("pretrained_model_sha256") != model_sha256:
        raise RuntimeError(f"Acceptance and transfer checkpoint diverge: {fold}")


sys.path.insert(0, str(runtime))
from contextual_pair_fusion import ContextualPairFusionAssociationModel

for fold in ACCEPTANCE_FOLDS:
    strict_probe = ContextualPairFusionAssociationModel()
    strict_probe.load_state_dict(
        torch.load(
            pretraining_root / fold / "pretrained_model.pt",
            map_location="cpu",
            weights_only=True,
        ),
        strict=True,
    )
    if sum(parameter.numel() for parameter in strict_probe.parameters()) != 20_747_761:
        raise RuntimeError(f"Accepted contextual checkpoint architecture changed: {fold}")
    del strict_probe
print("Both accepted v3 checkpoints strict-loaded for multiscale warm start.")


In [ ]:
command = [
    sys.executable,
    str(runtime / "train_zebrahub_multiscale_contextual_pretrain.py"),
    "--orchestrate",
    "--data-root", str(shard_root),
    "--output-dir", str(OUTPUT_DIR),
    "--initial-model-root", str(pretraining_root),
    "--seed", "51004",
    "--steps", "12000",
    "--minimum-train-shards", "64",
    "--minimum-validation-shards", "16",
    "--patch-batch-size", "32",
    "--gradient-accumulation", "3",
    "--validation-every", "500",
    "--learning-rate", "0.0002",
    "--minimum-learning-rate", "0.000002",
    "--weight-decay", "0.00001",
    "--ema-decay", "0.997",
    "--augmentation-mode", "microscopy_v1",
    "--max-wall-seconds", "21600",
    "--orchestrator-hard-stop-seconds", "22800",
    "--finalization-reserve-seconds", "1200",
]
print("Launching two-fold ZebraHub multiscale contextual pretraining:", " ".join(command))
try:
    subprocess.run(command, check=True)
except Exception as error:
    write_terminal("failed", error)
    raise

pretraining_terminal = OUTPUT_DIR / "pretraining_terminal.json"
if not pretraining_terminal.is_file():
    raise RuntimeError("Pretrainer exited without terminal evidence")
result = json.loads(pretraining_terminal.read_text(encoding="utf-8"))
folds = result.get("folds", {})
valid_folds = bool(
    set(folds) == {"target_44b6", "target_6bba"}
    and all(
        row.get("status") == "completed"
        and row.get("appearance_family")
            == "temporal_multiscale_contextual_pair_fusion_v4"
        and row.get("parameter_count") == 46_386_607
        and row.get("initialization", {}).get("run_id")
            == "zebrahub-contextual-pretrain-v1"
        and row.get("initialization", {}).get("source_family")
            == "temporal_contextual_pair_fusion_v3"
        and row.get("initialization", {}).get("target_family")
            == "temporal_multiscale_contextual_pair_fusion_v4"
        and row.get("initialization", {}).get(
            "initial_predictions_numerically_preserved"
        ) is True
        and row.get("dataset_manifest_sha256") == EXPECTED_DATASET_MANIFEST_SHA256
        and row.get("augmentation_mode") == "microscopy_v1"
        and row.get("validation_augmentation") == "none"
        and row.get("reciprocal_parent_loss_weight") == 0.35
        and row.get("shard_cache_policy")
            == "verified immutable tensors preloaded once per GPU"
        and row.get("validation_precision") == "CUDA float16 autocast"
        and row.get("selection_gate_passed") is True
        and row.get("audit_gate_passed") is True
        and int(row.get("best_step", 0)) > 0
        and row.get("validation_partition_policy")
            == (
                "ZSNS005 disjoint developmental windows: t0096-0099/t0376-0379 "
                "checkpoint selection; t0236-0239/t0516-0519 one-shot audit"
            )
        and row.get("competition_data_read") is False
        and row.get("public_predictions_copied") is False
        and row.get("public_leaderboard_used_for_selection") is False
        and row.get("submission_created") is False
        and (OUTPUT_DIR / fold / "pretrained_model.pt").is_file()
        and hashlib.sha256((OUTPUT_DIR / fold / "pretrained_model.pt").read_bytes()).hexdigest()
            == row.get("model_sha256")
        for fold, row in folds.items()
    )
)
if not (
    result.get("status") == "completed"
    and result.get("run_id") == PRETRAINING_RUN_ID
    and result.get("appearance_family")
        == "temporal_multiscale_contextual_pair_fusion_v4"
    and result.get("gpu_count") == 2
    and result.get("both_folds_improved") is True
    and result.get("augmentation_mode") == "microscopy_v1"
    and result.get("validation_augmentation") == "none"
    and result.get("competition_data_read") is False
    and result.get("public_predictions_copied") is False
    and result.get("public_leaderboard_used_for_selection") is False
    and result.get("submission_created") is False
    and valid_folds
):
    raise RuntimeError(f"Invalid pretraining terminal: {result}")
print(json.dumps(result, indent=2, sort_keys=True))


In [ ]:
FINISHED = True
TIMER.cancel()
write_terminal("completed")
print("Multiscale external pretraining complete; no competition data or submission path was used.")
